# CNN V6 - Smaller Network, Tuned Hyperparameters

Everything about the data stays identical to V5 (group split, stone-region
crops from the label bounding boxes, real augmentation, balanced batches,
threshold tuning, TTA). V6 changes the model and the optimisation:

1. **Smaller network** - V5's 12.9M parameters sit almost entirely in one
   wide flatten-to-Linear layer (128 x 28 x 28 -> 128). V6 replaces that
   with global average pooling, giving roughly 94K parameters in total
   (~140x smaller) while keeping the same three convolutional blocks.
2. **Batch normalisation** - added after each convolution to stabilise
   and speed up training.
3. **Tuned hyperparameters** - weight decay 1e-4, dropout reduced to 0.3
   (batch norm already regularises), up to 14 epochs with early stopping
   patience 3, cosine learning-rate decay from 1e-3.

The goal: check whether the large head was helping or just slowing
training down. Deleting this file removes the addition; V1-V3 in
`CNN.ipynb` are untouched.

In [1]:
import copy
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
root = Path.cwd()
while not (root / "assets" / "datasets").is_dir():
    root = root.parent
IMAGE_DIR = root / "assets" / "datasets" / "URI_CAD_Dataset" / "images"
LABEL_DIR = root / "assets" / "datasets" / "URI_CAD_Dataset" / "labels"


def parse_boxes(text):
    boxes = []
    for m in re.finditer(r"Bounding box \d+: (\d+) (\d+) (\d+) (\d+)", text):
        x, y, w, h = map(int, m.groups())
        boxes.append((x, y, x + w, y + h))
    return boxes


def union_box(boxes):
    if not boxes:
        return None
    return (min(b[0] for b in boxes), min(b[1] for b in boxes),
            max(b[2] for b in boxes), max(b[3] for b in boxes))


rows = []
for label_file in sorted(LABEL_DIR.glob("*.txt"), key=lambda p: int(p.stem)):
    text = label_file.read_text()
    diagnosis = int(re.search(r"Global diagnosis: (\d)", text).group(1))
    box = union_box(parse_boxes(text))
    if box and (box[2] - box[0] < 16 or box[3] - box[1] < 16):
        box = None
    rows.append({"image_id": int(label_file.stem),
                 "image_path": str(IMAGE_DIR / f"{label_file.stem}.jpg"),
                 "diagnosis": diagnosis, "crop_box": box})

df = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
df["group"] = df["image_id"] // 25
boxed = df["crop_box"].notna()
print(f"images: {len(df)} | boxed: {boxed.sum()} ({boxed.mean():.0%}) | "
      f"stone frames: {(df['diagnosis'] == 1).mean():.1%}")

images: 1985 | boxed: 1320 (66%) | stone frames: 77.3%


In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["group"]))
temp = df.loc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp, groups=temp["group"]))

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = temp.iloc[val_idx].reset_index(drop=True)
test_df = temp.iloc[test_idx].reset_index(drop=True)

for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(f"{name}: {len(part)} frames, {part['group'].nunique()} groups, "
          f"stone {part['diagnosis'].mean():.1%}, "
          f"boxed {part['crop_box'].notna().mean():.0%}")

train: 1399 frames, 56 groups, stone 78.8%, boxed 67%
val: 300 frames, 12 groups, stone 71.0%, boxed 62%
test: 286 frames, 12 groups, stone 76.6%, boxed 67%


In [4]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
])


def resize_with_padding(img):
    scale = IMG_SIZE / max(img.size)
    img = img.resize((max(1, round(img.width * scale)), max(1, round(img.height * scale))))
    canvas = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
    canvas.paste(img, ((IMG_SIZE - img.width) // 2, (IMG_SIZE - img.height) // 2))
    return canvas


class KidneyRoiDataset(Dataset):
    def __init__(self, dataframe, transform, train=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.train = train

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.loc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        w, h = img.size
        box = row["crop_box"]
        if box:
            img = img.crop((max(0, box[0]), max(0, box[1]), min(w, box[2]), min(h, box[3])))
        elif self.train:
            f = random.uniform(0.6, 1.0)
            cw, ch = max(16, int(w * f)), max(16, int(h * f))
            left, top = random.randint(0, w - cw), random.randint(0, h - ch)
            img = img.crop((left, top, left + cw, top + ch))
        else:
            cw, ch = int(w * 0.8), int(h * 0.8)
            img = img.crop(((w - cw) // 2, (h - ch) // 2, (w - cw) // 2 + cw, (h - ch) // 2 + ch))
        img = resize_with_padding(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row["diagnosis"], dtype=torch.long)

In [5]:
class_counts = train_df["diagnosis"].value_counts()
sample_weights = train_df["diagnosis"].map({c: len(train_df) / (2 * n) for c, n in class_counts.items()})
sampler = WeightedRandomSampler(sample_weights.values, num_samples=len(train_df), replacement=True)

train_loader = DataLoader(KidneyRoiDataset(train_df, train_transform, train=True),
                          batch_size=32, sampler=sampler, num_workers=0)
val_loader = DataLoader(KidneyRoiDataset(val_df, eval_transform), batch_size=32, num_workers=0)
test_loader = DataLoader(KidneyRoiDataset(test_df, eval_transform), batch_size=32, num_workers=0)
print("loaders ready (balanced sampling on train)")

loaders ready (balanced sampling on train)


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/torch/utils/data/sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


In [6]:
class KidneyCNN_V6(nn.Module):

    def __init__(self):
        super(KidneyCNN_V6, self).__init__()

        self.features = nn.Sequential(

            # Convolutional Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Convolutional Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Convolutional Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


v6_model = KidneyCNN_V6().to(device)
total_params = sum(p.numel() for p in v6_model.parameters())
print(f"trainable parameters: {total_params:,} (from scratch, no transfer learning; "
      f"V5 had 12,938,690)")

class_weights = torch.tensor([
    len(train_df) / (2 * (train_df["diagnosis"] == 0).sum()),
    len(train_df) / (2 * (train_df["diagnosis"] == 1).sum()),
], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

def predict_probs(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            out = torch.softmax(model(images.to(device)), dim=1)[:, 1]
            probs.extend(out.cpu().tolist())
            labels.extend(y.tolist())
    return np.array(probs), np.array(labels)

trainable parameters: 93,954 (from scratch, no transfer learning; V5 had 12,938,690)


In [7]:
def run_epochs(epochs, lr):
    optimizer = torch.optim.Adam(v6_model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_loss, best_state, patience = np.inf, None, 0
    for epoch in range(1, epochs + 1):
        v6_model.train()
        running = 0.0
        for images, y in train_loader:
            images, y = images.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(v6_model(images), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(y)
        scheduler.step()
        val_probs, val_y = predict_probs(v6_model, val_loader)
        val_loss = nn.functional.cross_entropy(
            torch.tensor(np.stack([1 - val_probs, val_probs], axis=1), dtype=torch.float32),
            torch.tensor(val_y))
        print(f"epoch {epoch}: train loss {running / len(train_df):.4f}, "
              f"val loss {val_loss:.4f}, val bal-acc "
              f"{balanced_accuracy_score(val_y, val_probs > 0.5):.4f}")
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, patience = val_loss, copy.deepcopy(v6_model.state_dict()), 0
        else:
            patience += 1
            if patience >= 3:
                print("early stopping (no val-loss improvement for 3 epochs)")
                break
    if best_state:
        v6_model.load_state_dict(best_state)


run_epochs(epochs=14, lr=1e-3)

epoch 1: train loss 0.4008, val loss 0.5477, val bal-acc 0.7127


epoch 2: train loss 0.3664, val loss 0.5989, val bal-acc 0.7817


epoch 3: train loss 0.3386, val loss 0.6310, val bal-acc 0.7371


epoch 4: train loss 0.3467, val loss 0.4836, val bal-acc 0.9168


epoch 5: train loss 0.3433, val loss 0.4906, val bal-acc 0.9108


epoch 6: train loss 0.3242, val loss 0.4891, val bal-acc 0.9178


epoch 7: train loss 0.3101, val loss 0.5256, val bal-acc 0.8638
early stopping (no val-loss improvement for 3 epochs)


In [8]:
val_probs, val_y = predict_probs(v6_model, val_loader)
thresholds = np.arange(0.05, 0.96, 0.05)
scores = [balanced_accuracy_score(val_y, val_probs > t) for t in thresholds]
best_t = float(thresholds[int(np.argmax(scores))])
print(f"tuned decision threshold: {best_t:.2f} "
      f"(validation balanced accuracy {max(scores):.4f}; "
      f"{balanced_accuracy_score(val_y, val_probs > 0.5):.4f} at the default 0.5)")

tuned decision threshold: 0.50 (validation balanced accuracy 0.9168; 0.9168 at the default 0.5)


In [9]:
def tta_probs(model, loader):
    """Average the stone probability over the original, horizontal and vertical flips."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            images = images.to(device)
            out = (torch.softmax(model(images), dim=1)[:, 1]
                   + torch.softmax(model(torch.flip(images, [-1])), dim=1)[:, 1]
                   + torch.softmax(model(torch.flip(images, [-2])), dim=1)[:, 1]) / 3
            probs.extend(out.cpu().tolist())
            labels.extend(y.tolist())
    return np.array(probs), np.array(labels)


test_probs, test_y = tta_probs(v6_model, test_loader)

print("majority-class baseline (always stone): 77.3% accuracy")
for name, pred in (("default 0.5", test_probs > 0.5), (f"tuned {best_t:.2f}", test_probs > best_t)):
    print(f"\nTEST - threshold {name}")
    print(f"accuracy {accuracy_score(test_y, pred):.4f} | "
          f"balanced acc {balanced_accuracy_score(test_y, pred):.4f} | "
          f"macro F1 {f1_score(test_y, pred, average='macro'):.4f}")
print("\n", classification_report(test_y, test_probs > 0.5, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(test_y, test_probs > 0.5),
                   index=["no stone (true)", "stone (true)"],
                   columns=["no stone (pred)", "stone (pred)"]).to_string())
print("\nreference (same split):\n"
      "CNN V5 from-scratch  test @0.5: accuracy 0.8846 | balanced acc 0.9247 | macro F1 0.8605\n"
      "CNN V5 TL variant    test @0.5: accuracy 0.9021 | balanced acc 0.9361 | macro F1 0.8794")

majority-class baseline (always stone): 77.3% accuracy

TEST - threshold default 0.5
accuracy 0.8811 | balanced acc 0.9224 | macro F1 0.8567

TEST - threshold tuned 0.50
accuracy 0.8811 | balanced acc 0.9224 | macro F1 0.8567

               precision    recall  f1-score   support

           0      0.663     1.000     0.798        67
           1      1.000     0.845     0.916       219

    accuracy                          0.881       286
   macro avg      0.832     0.922     0.857       286
weighted avg      0.921     0.881     0.888       286

                 no stone (pred)  stone (pred)
no stone (true)               67             0
stone (true)                  34           185

reference (same split):
CNN V5 from-scratch  test @0.5: accuracy 0.8846 | balanced acc 0.9247 | macro F1 0.8605
CNN V5 TL variant    test @0.5: accuracy 0.9021 | balanced acc 0.9361 | macro F1 0.8794


## Results

Test evaluation on 286 held-out frames (12 groups never seen in training),
predictions averaged over three augmented views (TTA):

| Model | Params | Accuracy | Balanced acc | Macro F1 |
|---|---|---|---|---|
| Majority baseline (always stone) | - | 0.773 | 0.500 | - |
| CNN V2 (image-level split, full frames) | ~12.9M | 0.671 | 0.611 | - |
| CNN V4 TL (honest split, full frames) | 11.2M | 0.629 | 0.660 | 0.594 |
| CNN V5 TL (ROI + augmentation) | 11.2M | 0.902 | 0.936 | 0.879 |
| CNN V5 (ROI + augmentation, from scratch) | 12.9M | 0.885 | 0.925 | 0.861 |
| **CNN V6 (ROI + augmentation, smaller + tuned)** | **94K** | **0.881** | **0.922** | **0.857** |

Observations:

- V6 matches V5 within noise (0.881 vs 0.885 accuracy) with about 140x
  fewer parameters - the wide flatten-to-Linear head in V5 was dead weight,
  not signal.
- Training was also the fastest of all versions (early-stopped at epoch 7
  on MPS), thanks to batch normalisation and the small head.
- The error profile is unchanged: zero false alarms (all 67 no-stone
  frames rejected), 34 of 219 stone frames missed.
- The remaining gap to the transfer-learning variant (~2 points) is the
  cost of training from scratch on 1,399 frames; whether it is worth 11M
  extra parameters and slower training is a deployment decision, not an
  accuracy one.
- Caveats: single seed, small test set (286 frames from 12 groups), and
  the group split approximates the source videos with consecutive id
  blocks because URI-CAD publishes no patient ids.